In [ ]:
import json

RESULT_PATH = "/FINAL_PROJECT/results/experiment_4/gemma/Fewshot_CoT_experiment_2/result.json"
DEBUG_PATH = "/FINAL_PROJECT/results/experiment_4/gemma/Fewshot_CoT_experiment_2/debug_info.json"

with open(RESULT_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

with open(DEBUG_PATH, "r", encoding="utf-8") as f:
    debug_info = json.load(f)

# Build lookup: sent_id -> raw_response
debug_map = {s["sent_id"]: s["raw_response"] for s in debug_info["samples"]}

# === FIND CONTRADICTIONS ===
# Mâu thuẫn: result có opinions rỗng BUT raw_response không rỗng
contradictions = []

for sample in results:
    sent_id = sample["sent_id"]
    opinions = sample.get("opinions", [])

    # Chỉ quan tâm sample có opinions rỗng trong result
    if len(opinions) > 0:
        continue

    raw_response = debug_map.get(sent_id, "")

    # Mâu thuẫn: model đã trả lời nhưng result lại rỗng
    if raw_response.strip():
        contradictions.append({
            "sent_id": sent_id,
            "raw_response": raw_response
        })

# === REPORT ===
print(f"Total samples         : {len(results)}")
print(f"Contradictions found  : {len(contradictions)}")
print(f"Contradiction sent_ids: {[c['sent_id'] for c in contradictions]}")



Total samples         : 1085
Contradictions found  : 7
Contradiction sent_ids: [8327, 5556, 5733, 5904, 5913, 5923, 1703]

Saved to contradictions.json


In [9]:
import json
import sys
sys.path.append("/FINAL_PROJECT")

from src.utils.postprocessing import extract_json_from_response, postprocess_response

# Load contradictions
with open("contradictions.json", "r", encoding="utf-8") as f:
    contradictions = json.load(f)

# Trace từng sample qua đúng pipeline thực tế
print("=" * 70)
for item in contradictions:
    sent_id = item["sent_id"]
    raw = item["raw_response"]
    
    print(f"\n[sent_id={sent_id}]")
    
    # Step 1: extract_json_from_response (giống pipeline thực tế)
    extracted = extract_json_from_response(raw)
    print(f"  >> extracted (first 200): {extracted[:200]}")
    
    # Step 2: postprocess_response (giống pipeline thực tế)
    try:
        final = postprocess_response(extracted, "dummy_text", sent_id)
        parsed = json.loads(final)
        print(f"  >> opinions count: {len(parsed.get('opinions', []))}")
    except Exception as e:
        print(f"  ❌ postprocess failed: {e}")
    
    print("-" * 70)


[sent_id=8327]
  >> extracted (first 200): json
{
  "sent_id": 8327,
  "text": "Có tiền, có xe mà ý thức thua điểm thi hóa của cháu nữa :\\\"\\\(((\"",
  "opinions": [
    {
      "Source": [],
      "Target": ["cháu"],
      "Polar_expression
  >> opinions count: 0
----------------------------------------------------------------------

[sent_id=5556]
  >> extracted (first 200): json
{
  "sent_id": 5556,
  "text": """Nghe nói em đẹp tự nhiên không có sửa ha” Ừ thì em có sửa gì đâu, em chỉ niềng răng thôi, còn muốn biết niềng ở đâu xịn thì vô đây, link nè:""",
  "opinions": [

  >> opinions count: 0
----------------------------------------------------------------------

[sent_id=5733]
  >> extracted (first 200): json
{
  "sent_id": 5733,
  "text": "IQ vô cực :(( Đó là baroibeo, còn 3Ga thì Âm vô cực :(("" ,
  "opinions": [
    {
      "Source": [],
      "Target": ["baroibeo"],
      "Polar_expression": [
   
  >> opinions count: 0
------------------------------------------------------

In [10]:
import json
import re
from typing import Any


def extract_position(text: str, expression: str) -> str:
    start = text.find(expression)
    if start == -1:
        return "0:0"
    end = start + len(expression)
    return f"{start}:{end}"


def _extract_by_fields(text: str) -> str:
    """Last resort: build JSON bằng regex extraction từng field."""
    sent_id_m = re.search(r'"sent_id"\s*:\s*(\d+)', text)
    text_m = re.search(r'"text"\s*:\s*"((?:[^"\\]|\\.)*)"', text)
    result = {
        "sent_id": int(sent_id_m.group(1)) if sent_id_m else None,
        "text": text_m.group(1) if text_m else "",
        "opinions": []
    }
    opinions_start = text.find('"opinions"')
    if opinions_start != -1:
        arr_start = text.find('[', opinions_start)
        if arr_start != -1:
            depth, obj_start = 0, None
            for i in range(arr_start, len(text)):
                if text[i] == '{':
                    if depth == 0:
                        obj_start = i
                    depth += 1
                elif text[i] == '}':
                    depth -= 1
                    if depth == 0 and obj_start is not None:
                        obj_text = text[obj_start:i+1]
                        try:
                            result["opinions"].append(json.loads(obj_text))
                        except Exception:
                            pol_m = re.search(r'"Polarity"\s*:\s*"(\w+)"', obj_text)
                            int_m = re.search(r'"Intensity"\s*:\s*"(\w+)"', obj_text)
                            if pol_m:
                                result["opinions"].append({
                                    "Source": [], "Target": [], "Polar_expression": [],
                                    "Polarity": pol_m.group(1),
                                    "Intensity": int_m.group(1) if int_m else ""
                                })
                        obj_start = None
    return json.dumps(result, ensure_ascii=False)


def extract_json_from_response(raw_response: str) -> str:
    """
    Extract JSON từ raw response của LLM với xử lý robust.

    Các lỗi được fix:
    1. Bug regex cũ (Strategy 1): dùng ký tự invisible thay vì backtick thật
    2. Invalid/over-escaped sequences: \\\\\\\" -> \\"
    3. Curly/unicode quotes: " " → straight quotes
    4. Triple-double-quotes bao quanh text field: \"\"\"...\"\"\"
    5. Trailing extra quote: \"\"  ,  ->  \" ,
    6. Unclosed Polar_expression array
    7. Fallback field-by-field extraction
    """
    response = raw_response.strip()

    # Bước 1: Extract block giữa ```json...``` (fix bug backtick invisible)
    match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', response)
    if match:
        candidate = match.group(1).strip()
    else:
        candidate = ""
        first_brace = response.find('{')
        if first_brace != -1:
            open_braces = 0
            for i in range(first_brace, len(response)):
                if response[i] == '{':
                    open_braces += 1
                elif response[i] == '}':
                    open_braces -= 1
                    if open_braces == 0:
                        candidate = response[first_brace:i+1]
                        break
        if not candidate:
            return response.replace('```', '').strip()

    # Bước 2: Parse trực tiếp
    try:
        json.loads(candidate)
        return candidate
    except json.JSONDecodeError:
        pass

    # Bước 3: Fix lần lượt các lỗi phổ biến
    fixed = candidate

    # Fix curly/unicode quotes
    fixed = fixed.replace('\u201c', '\\"').replace('\u201d', '\\"')
    fixed = fixed.replace('\u2018', "\\'").replace('\u2019', "\\'")

    # Fix over-escaped: 3+ backslashes trước quote -> \"
    fixed = re.sub(r'\\{3,}"', '\\\\"', fixed)

    # Fix triple-double-quotes: "text": """..."""
    def fix_triple_quotes(m):
        inner = m.group(1).replace('"', '\\"')
        return f'"text": "{inner}"'
    fixed = re.sub(r'"text":\s*"""([\s\S]*?)"""', fix_triple_quotes, fixed)

    # Fix double outer quotes: "text": ""...""
    fixed = re.sub(
        r'"text":\s*""([\s\S]*?)""(?=\s*[,}])',
        lambda m: f'"text": "{m.group(1).replace(chr(34), chr(92)+chr(34))}"',
        fixed
    )

    # Fix trailing extra quote: ""  ,  ->  " ,
    fixed = re.sub(r'""\s*([,}\]])', r'"\1', fixed)

    # Fix unclosed Polar_expression array
    fixed = re.sub(r'(\["[^"\]]*")\s*\n(\s*"Polarity")', r'\1]\n\2', fixed)

    try:
        json.loads(fixed)
        return fixed
    except json.JSONDecodeError:
        pass

    # Bước 4: Last resort - field-by-field extraction
    return _extract_by_fields(candidate)


def postprocess_response(response_text: str, original_text: str, sent_id: Any) -> str:
    """Normalize model response to SemEval format."""
    try:
        result = json.loads(response_text)
        result["sent_id"] = sent_id
        result["text"] = original_text

        if "opinions" in result and isinstance(result["opinions"], list):
            for opinion in result["opinions"]:
                if not isinstance(opinion, dict):
                    continue

                for component in ["Source", "Target", "Polar_expression"]:
                    if component not in opinion:
                        opinion[component] = [[], []]
                        continue
                    component_data = opinion[component]
                    if isinstance(component_data, list):
                        if (len(component_data) == 2 and
                                isinstance(component_data[0], list) and
                                isinstance(component_data[1], list)):
                            texts = component_data[0]
                        else:
                            texts = component_data
                        valid_texts = [t for t in texts if isinstance(t, str) and t.strip()]
                        positions = [extract_position(original_text, t) for t in valid_texts]
                        opinion[component] = [valid_texts, positions]
                    elif isinstance(component_data, str) and component_data.strip():
                        t = component_data.strip()
                        opinion[component] = [[t], [extract_position(original_text, t)]]
                    else:
                        opinion[component] = [[], []]

                valid_polarities = ["Positive", "Negative", "Neutral"]
                if "Polarity" not in opinion or opinion["Polarity"] not in valid_polarities:
                    opinion["Polarity"] = ""

                valid_intensities = ["Strong", "Standard", "Weak"]
                if "Intensity" not in opinion or opinion["Intensity"] not in valid_intensities:
                    opinion["Intensity"] = ""
        else:
            result["opinions"] = []

        return json.dumps(result, ensure_ascii=False, indent=2)

    except json.JSONDecodeError:
        return json.dumps({"sent_id": sent_id, "text": original_text, "opinions": []},
                          ensure_ascii=False, indent=2)

In [11]:
import json
import sys
sys.path.append("/FINAL_PROJECT")

# Load contradictions
with open("contradictions.json", "r", encoding="utf-8") as f:
    contradictions = json.load(f)

# Trace từng sample qua đúng pipeline thực tế
print("=" * 70)
for item in contradictions:
    sent_id = item["sent_id"]
    raw = item["raw_response"]
    
    print(f"\n[sent_id={sent_id}]")
    
    # Step 1: extract_json_from_response (giống pipeline thực tế)
    extracted = extract_json_from_response(raw)
    print(f"  >> extracted (first 200): {extracted[:200]}")
    
    # Step 2: postprocess_response (giống pipeline thực tế)
    try:
        final = postprocess_response(extracted, "dummy_text", sent_id)
        parsed = json.loads(final)
        print(f"  >> opinions count: {len(parsed.get('opinions', []))}")
    except Exception as e:
        print(f"  ❌ postprocess failed: {e}")
    
    print("-" * 70)


[sent_id=8327]
  >> extracted (first 200): {"sent_id": 8327, "text": "Có tiền, có xe mà ý thức thua điểm thi hóa của cháu nữa :\\\\\\\"\\\\\\(((\\\"", "opinions": [{"Source": [], "Target": ["cháu"], "Polar_expression": ["Có tiền, có xe", "ý th
  >> opinions count: 1
----------------------------------------------------------------------

[sent_id=5556]
  >> extracted (first 200): {"sent_id": 5556, "text": "", "opinions": [{"Source": [], "Target": ["em"], "Polar_expression": ["đẹp tự nhiên không có sửa"], "Polarity": "Positive", "Intensity": "Standard"}, {"Source": [], "Target"
  >> opinions count: 3
----------------------------------------------------------------------

[sent_id=5733]
  >> extracted (first 200): {
  "sent_id": 5733,
  "text": "IQ vô cực :(( Đó là baroibeo, còn 3Ga thì Âm vô cực :((",
  "opinions": [
    {
      "Source": [],
      "Target": ["baroibeo"],
      "Polar_expression": [
        "I
  >> opinions count: 2
------------------------------------------------------

In [12]:
import json
import re
from pathlib import Path


# =========================================================
# Check if opinions exist after extraction
# =========================================================
def has_opinions_after_extract(raw_response: str) -> bool:

    extracted = extract_json_from_response(raw_response)

    try:
        parsed = json.loads(extracted)
        return len(parsed.get("opinions", [])) > 0
    except Exception:
        return False


# =========================================================
# Scan experiment results
# =========================================================
EXPERIMENT_ROOT = Path("/FINAL_PROJECT/results/experiment_1")

before_total = 0
after_total = 0

remaining = []

for model_dir in sorted(EXPERIMENT_ROOT.iterdir()):

    if not model_dir.is_dir():
        continue

    for exp_dir in sorted(model_dir.iterdir()):

        if not exp_dir.is_dir():
            continue

        result_path = exp_dir / "result.json"
        debug_path = exp_dir / "debug_info.json"

        if not result_path.exists() or not debug_path.exists():
            continue

        try:

            with open(result_path, "r", encoding="utf-8") as f:
                results = json.load(f)

            with open(debug_path, "r", encoding="utf-8") as f:
                debug_info = json.load(f)

        except Exception as e:
            print(f"⚠️ Load error [{exp_dir.name}]: {e}")
            continue

        debug_map = {
            s["sent_id"]: s["raw_response"]
            for s in debug_info.get("samples", [])
        }

        folder_key = f"{model_dir.name}/{exp_dir.name}"

        exp_before = 0
        exp_after = 0

        for sample in results:

            sent_id = sample["sent_id"]

            if len(sample.get("opinions", [])) > 0:
                continue

            raw = debug_map.get(sent_id, "")

            if not raw.strip():
                continue

            exp_before += 1
            before_total += 1

            if not has_opinions_after_extract(raw):

                exp_after += 1
                after_total += 1

                remaining.append(
                    {
                        "folder": folder_key,
                        "sent_id": sent_id,
                    }
                )

        if exp_before > 0:
            print(
                f"[{folder_key}] Before: {exp_before} | After fix: {exp_after} remaining"
            )


# =========================================================
# Report
# =========================================================
print("\n" + "=" * 60)

print(f"TOTAL contradictions (before fix): {before_total}")
print(f"TOTAL still failing  (after fix) : {after_total}")
print(f"Fixed                            : {before_total - after_total}")

print("=" * 60)


# =========================================================
# Save failures
# =========================================================
with open("remaining_failures.json", "w", encoding="utf-8") as f:
    json.dump(remaining, f, ensure_ascii=False, indent=2)

print(
    f"\nSaved {len(remaining)} remaining failures to remaining_failures.json"
)

[bloomvn/Fewshot_experiment_1] Before: 313 | After fix: 299 remaining
[bloomvn/Fewshot_experiment_2] Before: 238 | After fix: 223 remaining
[bloomvn/Fewshot_experiment_3] Before: 315 | After fix: 292 remaining
[bloomvn/Fewshot_experiment_4] Before: 307 | After fix: 282 remaining
[bloomvn/Re_reading_experiment_1] Before: 469 | After fix: 443 remaining
[gemma/Fewshot_CoT_experiment_1] Before: 3 | After fix: 0 remaining
[gemma/Fewshot_CoT_experiment_2] Before: 4 | After fix: 0 remaining
[gemma/Fewshot_CoT_experiment_3] Before: 2 | After fix: 0 remaining
[gemma/Fewshot_CoT_experiment_4] Before: 3 | After fix: 0 remaining
[gemma/Fewshot_experiment_1] Before: 3 | After fix: 0 remaining
[gemma/Fewshot_experiment_2] Before: 3 | After fix: 0 remaining
[gemma/Fewshot_experiment_3] Before: 8 | After fix: 0 remaining
[gemma/Fewshot_experiment_4] Before: 7 | After fix: 0 remaining
[gemma/Fewshot_experiment_6] Before: 3 | After fix: 0 remaining
[gemma/PS_CoT_experiment_1] Before: 34 | After fix: 0 r

In [18]:
raw_response="Tuyệt vời! Chúng ta sẽ cùng thực hiện phân tích cảm xúc cho bình luận này theo quy trình chi tiết bạn đã đề ra.\n\n**1. Phân tích Sơ bộ & Trích xuất Dấu hiệu:**\n\n*   **Văn bản:** \"sau này nó lớn lên làm kinh doanh thứ gì đó thì lúc đó nó mới biết thế nào là quả báo\"\n*   **Dấu hiệu:**\n    *   “nó”: Đại từ nhân xưng (Target - có thể ám chỉ một người cụ thể)\n    *   “sau này”: Chỉ thời gian, mang tính dự đoán\n    *   “làm kinh doanh”: Hành động, thể hiện một hoạt động cụ thể\n    *   “quả báo”: Cụm từ mang tính ẩn ý, gợi ý về hậu quả, sự trừng phạt\n    *   “biết thế nào là”: Thể hiện sự nhận thức, hiểu biết\n\n**2. Lập Kế hoạch Phân tích:**\n\nChúng ta sẽ tập trung vào một Opinion duy nhất trong câu này:  ý kiến về hậu quả mà \"nó\" sẽ gặp phải sau khi làm kinh doanh.  Opinion này mang sắc thái tiêu cực, ám chỉ một sự trừng phạt hoặc hậu quả không mong muốn.\n\n*   **Source:** Không có đại từ nhân xưng trực tiếp.  \"Nó\" có thể ám chỉ một người cụ thể, nhưng không được chỉ định rõ ràng.\n*   **Target:** \"Nó\" (người được ám chỉ)\n*   **Polar_expression:** \"quả báo\" (cụm từ mang ý nghĩa tiêu cực)\n*   **Polarity:** Negative (do \"quả báo\" thường liên quan đến hậu quả xấu)\n*   **Intensity:** Standard (cường độ không quá mạnh, chỉ là một dự đoán về hậu quả)\n\n**3. Thực thi Kế hoạch:**\n\nDựa trên phân tích trên, chúng ta có thể xây dựng một Opinion như sau:\n\n```json\n{\n  \"Source\": null,\n  \"Target\": \"nó\",\n  \"Polar_expression\": \"quả báo\",\n  \"Polarity\": \"Negative\",\n  \"Intensity\": \"Standard\"\n}\n```\n\n**Giải thích:**\n\n*   **Source: null** - Không có đại từ nhân xưng trực tiếp.\n*   **Target: \"nó\"** -  Đối tượng hướng tới là người được ám chỉ.\n*   **Polar_expression: \"quả báo\"** -  Đây là biểu thức cảm xúc chính, thể hiện ý kiến về hậu quả.\n*   **Polarity: \"Negative\"** -  \"Quả báo\" thường mang ý nghĩa tiêu cực, báo hiệu điều không tốt.\n*   **Intensity: \"Standard\"** -  Cường độ không quá mạnh, chỉ là một dự đoán về hậu quả.\n\n**4. Tổng hợp Kết quả:**\n\n```json\n{\n  \"sent_id\": \"1534\",\n  \"text\": \"sau này nó lớn lên làm kinh doanh thứ gì đó thì lúc đó nó mới biết thế nào là quả báo\",\n  \"opinions\": [\n    {\n      \"Source\": null,\n      \"Target\": \"nó\",\n      \"Polar_expression\": \"quả báo\",\n      \"Polarity\": \"Negative\",\n      \"Intensity\": \"Standard\"\n    }\n  ]\n}\n```\n\n**Lưu ý:**\n\n*   Đây là một phân tích dựa trên ngữ cảnh và hiểu biết chung về tiếng Việt.  Nếu có thêm thông tin về người nói, người nghe, hoặc bối cảnh cụ thể, phân tích có thể được điều chỉnh.\n*   Việc xác định \"Source\" có thể phức tạp hơn trong một số trường hợp, đặc biệt khi sử dụng các đại từ mơ hồ.\n\nBạn có muốn tôi phân tích một bình luận khác không?"


In [19]:
extracted = extract_json_from_response(raw_response)

In [20]:
extracted

'{\n  "Source": null,\n  "Target": "nó",\n  "Polar_expression": "quả báo",\n  "Polarity": "Negative",\n  "Intensity": "Standard"\n}'

In [21]:
postprocess_response(extracted, "dummy_text", 1534)

'{\n  "Source": null,\n  "Target": "nó",\n  "Polar_expression": "quả báo",\n  "Polarity": "Negative",\n  "Intensity": "Standard",\n  "sent_id": 1534,\n  "text": "dummy_text",\n  "opinions": []\n}'